# RPS II — The Same Agents in a Network

## Connectivity is hidden by averages

RPS I showed that repeated random play reproduces the analytical probabilities:

$$P(\text{win})=P(\text{loss})=P(\text{tie})=\frac{1}{3}.$$

In RPS II, we change **only who can interact with whom**. The agents, behavioral rule, payoff rule, and score update remain unchanged.

## Learning objectives

By the end of the notebook, you should be able to:

1. distinguish agent behavior from interaction structure;
2. explain why aggregate averages describe outcomes but may obscure mechanisms;
3. compare outcomes across networks while holding agents and rules constant;
4. connect the experiment to Axtell's Use II and Macy and Willer's actor-centered sociology.

## 1. What changes from RPS I?

| Component | RPS I | RPS II |
|---|---|---|
| Agents | Random RPS players | **Unchanged** |
| Behavioral rule | Random move | **Unchanged** |
| Payoff rule | Standard RPS | **Unchanged** |
| State update | Add points to scores | **Unchanged** |
| Interaction | Random pairing | **Network neighbors** |

This is the controlled comparison:

$$	ext{same agents}+	ext{same rules}+	ext{different connectivity}ightarrow	ext{different social structure of outcomes}.$$

## 2. The RPS I agents and rules

The following code is retained from RPS I. An agent still knows nothing about the network. It simply chooses a move randomly whenever it plays.

In [ ]:
from random import Random
import pandas as pd
import matplotlib.pyplot as plt
import math

MOVES = ['Rock', 'Paper', 'Scissors']

PAYOFF = {
    ('Rock', 'Paper'): (0, 1),
    ('Paper', 'Rock'): (1, 0),
    ('Rock', 'Scissors'): (1, 0),
    ('Scissors', 'Rock'): (0, 1),
    ('Paper', 'Scissors'): (0, 1),
    ('Scissors', 'Paper'): (1, 0),
    ('Rock', 'Rock'): (0, 0),
    ('Paper', 'Paper'): (0, 0),
    ('Scissors', 'Scissors'): (0, 0)
}

In [ ]:
def initialize_society(n_agents):
    return [
        {
            'name': f'Agent_{i:02d}',
            'score': 0,
            'move': None,
            'decision_rule': 'random'
        }
        for i in range(n_agents)
    ]


def choose_move(agent, rng):
    if agent['decision_rule'] == 'random':
        return rng.choice(MOVES)
    raise ValueError(f"Unknown decision rule: {agent['decision_rule']}")


def classify_outcome(points):
    if points == (1, 0):
        return 'win'
    if points == (0, 1):
        return 'loss'
    return 'tie'


def play_game(player1, player2, rng):
    player1['move'] = choose_move(player1, rng)
    player2['move'] = choose_move(player2, rng)

    points = PAYOFF[(player1['move'], player2['move'])]
    player1['score'] += points[0]
    player2['score'] += points[1]

    return {
        'player1': player1['name'],
        'move1': player1['move'],
        'player2': player2['name'],
        'move2': player2['move'],
        'points1': points[0],
        'points2': points[1],
        'outcome1': classify_outcome(points)
    }

## 3. Connectivity becomes part of the environment

We compare two societies of 31 agents:

- **Ring:** every agent has exactly two neighbors.
- **Star:** one hub is connected to all other agents; peripheral agents connect only to the hub.

The networks contain almost the same number of links. This helps us hold the total volume of interaction approximately constant while changing its distribution.

In [ ]:
N_AGENTS = 31

def ring_network(n_agents):
    return {
        'n_agents': n_agents,
        'links': [(i, (i + 1) % n_agents) for i in range(n_agents)]
    }


def star_network(n_agents):
    return {
        'n_agents': n_agents,
        'links': [(0, i) for i in range(1, n_agents)]
    }


def degrees(graph):
    result = {i: 0 for i in range(graph['n_agents'])}
    for i, j in graph['links']:
        result[i] += 1
        result[j] += 1
    return result


networks = {
    'ring': ring_network(N_AGENTS),       # 31 links
    'star': star_network(N_AGENTS)        # 30 links
}

pd.DataFrame([
    {
        'network': name,
        'agents': graph['n_agents'],
        'links': len(graph['links']),
        'average_degree': sum(degrees(graph).values()) / graph['n_agents'],
        'minimum_degree': min(degrees(graph).values()),
        'maximum_degree': max(degrees(graph).values())
    }
    for name, graph in networks.items()
])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

def circular_positions(n_agents):
    return {
        i: (math.cos(2 * math.pi * i / n_agents),
            math.sin(2 * math.pi * i / n_agents))
        for i in range(n_agents)
    }


positions = {
    'ring': circular_positions(N_AGENTS),
    'star': {0: (0, 0), **{
        i: (math.cos(2 * math.pi * (i - 1) / (N_AGENTS - 1)),
            math.sin(2 * math.pi * (i - 1) / (N_AGENTS - 1)))
        for i in range(1, N_AGENTS)
    }}
}

for ax, (name, graph) in zip(axes, networks.items()):
    position = positions[name]
    for i, j in graph['links']:
        ax.plot(
            [position[i][0], position[j][0]],
            [position[i][1], position[j][1]],
            color='lightgray', linewidth=1, zorder=1
        )
    ax.scatter(
        [position[i][0] for i in range(graph['n_agents'])],
        [position[i][1] for i in range(graph['n_agents'])],
        s=90, color='steelblue', zorder=2
    )
    ax.set_title(f'{name.title()} network')
    ax.set_aspect('equal')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 4. The network interaction rule

During one round, each network link is activated once. Connected agents play the same game used in RPS I. Links are shuffled so that execution order does not introduce a fixed sequence.

Notice that the network selects partners; it does not change how agents choose Rock, Paper, or Scissors.

In [ ]:
def play_network_round(society, graph, rng, round_number):
    links = list(graph['links'])
    rng.shuffle(links)

    events = []
    for i, j in links:
        event = play_game(society[i], society[j], rng)
        event['round'] = round_number
        events.append(event)

    return events


def simulate_network(graph, n_rounds=300, seed=123):
    rng = Random(seed)
    society = initialize_society(graph['n_agents'])
    history = []

    for round_number in range(1, n_rounds + 1):
        history.extend(
            play_network_round(society, graph, rng, round_number)
        )

    return society, pd.DataFrame(history)

## 5. Run the controlled experiment

Both societies use the same number of agents, rounds, decision rule, and payoff rule. Only the links differ.

In [ ]:
simulations = {}

for name, graph in networks.items():
    society, history = simulate_network(graph, n_rounds=300, seed=123)
    simulations[name] = {
        'graph': graph,
        'society': society,
        'history': history
    }

## 6. First look: aggregate averages

At the aggregate level, the two societies look very similar. Their win, loss, and tie frequencies remain close to the theoretical value of one third.

In [ ]:
aggregate_outcomes = pd.concat([
    result['history']['outcome1']
    .value_counts(normalize=True)
    .reindex(['win', 'loss', 'tie'])
    .rename(name)
    for name, result in simulations.items()
], axis=1)

aggregate_outcomes['theoretical'] = 1 / 3
aggregate_outcomes

In [ ]:
ax = aggregate_outcomes.plot.bar(figsize=(9, 4), ylim=(0, 0.5))
ax.set_title('Aggregate outcomes remain close to the RPS benchmark')
ax.set_xlabel('Outcome for Player 1')
ax.set_ylabel('Proportion')
ax.tick_params(axis='x', rotation=0)
plt.show()

If we stopped here, we might conclude that connectivity made little difference. The average is not wrong; it is **insufficient**.

## 7. Recover the agents behind the average

We now combine each agent's final score with its degree—the number of network neighbors—and its number of games.

In [ ]:
def agent_results(network_name, result):
    society = result['society']
    history = result['history']
    graph = result['graph']

    games = pd.concat([
        history['player1'],
        history['player2']
    ]).value_counts()

    table = pd.DataFrame(society)
    table['agent_id'] = range(len(table))
    table['network'] = network_name
    table['degree'] = table['agent_id'].map(degrees(graph))
    table['games'] = table['name'].map(games)
    table['score_per_game'] = table['score'] / table['games']
    return table


agent_tables = pd.concat([
    agent_results(name, result)
    for name, result in simulations.items()
], ignore_index=True)

agent_tables.head()

In [ ]:
network_summary = (
    agent_tables.groupby('network')
    .agg(
        mean_score=('score', 'mean'),
        median_score=('score', 'median'),
        minimum_score=('score', 'min'),
        maximum_score=('score', 'max'),
        mean_score_per_game=('score_per_game', 'mean')
    )
    .round(3)
)

network_summary

The mean scores are fairly similar because the networks have almost the same number of links. But their score distributions are radically different.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)

for ax, name in zip(axes, ['ring', 'star']):
    subset = (
        agent_tables[agent_tables['network'] == name]
        .sort_values('degree')
        .reset_index(drop=True)
    )
    unequal_degrees = subset['degree'].max() > subset['degree'].min()
    colors = [
        'firebrick' if unequal_degrees and degree == subset['degree'].max()
        else 'steelblue'
        for degree in subset['degree']
    ]
    ax.bar(range(len(subset)), subset['score'], color=colors)
    ax.axhline(subset['score'].mean(), color='black', linestyle='--',
               label='Network mean')
    ax.set_title(f'{name.title()}: individual scores')
    ax.set_xlabel('Agents ordered by degree')
    ax.legend()

axes[0].set_ylabel('Total score')
plt.tight_layout()
plt.show()

In the ring, all agents have the same connectivity and roughly comparable opportunities to score. In the star, the hub plays against every peripheral agent. Its high total score does not come from a better decision rule; it comes from its structural position.

In [ ]:
agent_tables.groupby('network')[['degree', 'games', 'score', 'score_per_game']].corr().round(3)

## 8. What the average hides

RPS I asks whether simulated aggregate frequencies reproduce a known probability. For that purpose, averages inform us very well.

RPS II asks how interactions are distributed across a social structure. Aggregate frequencies may remain near one third while hiding:

- who had opportunities to interact;
- who occupied a central or peripheral position;
- how unequally scores were distributed;
- which local interactions generated the aggregate result.

Therefore:

$$	ext{RPS I: averages summarize the result well}.$$

$$	ext{RPS II: averages summarize the result but obscure the mechanism}.$$

## 9. Axtell and Macy–Willer

**Axtell's Use II:** mathematics gives us the individual game and its expected aggregate probabilities. Agent computation complements that analysis when we embed those games in structured interaction networks and inspect the resulting configurations.

**Macy and Willer:** connectivity is a social factor, but it operates through actors. The network determines possible encounters; encounters produce individual scores; those scores generate the aggregate distribution.

$$	ext{connectivity}ightarrow	ext{encounters}ightarrow	ext{agent outcomes}ightarrow	ext{aggregate pattern}.$$

The agents have not become more intelligent. The model has become more social because interaction is now structured.

## Questions for discussion

1. Why do the aggregate outcome frequencies remain close to one third in both networks?
2. Why is the hub's high score not evidence of superior ability?
3. Which statistic reveals connectivity better: mean score, score per game, or the distribution of total scores?
4. In what sense does the network act as a factor through actors?
5. What information would disappear if we retained only the aggregate-outcome table?